# 01 · Data Audit
**Goal:** understand both raw datasets *before* touching them, covering structure, sampling interval, missing data, value ranges and site location. The output is a decision on which dataset plays which role.

Run each cell with **Shift + Enter**, from top to bottom.

In [1]:
import sys, warnings
sys.path.insert(0, "../src")          # project modules
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

from io_utils import RAW, load_primary_raw, load_site_a_raw

## 1. Primary dataset: `solar_dataset.csv`
First look at the raw text. **Every line is wrapped in double quotes**, so a plain `pd.read_csv` would load each row as a single column.

In [2]:
with open(RAW / "solar_dataset.csv", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().rstrip())

"timestamp,Active_Energy_Delivered_Received,Current_Phase_Average,Active_Power,Wind_Speed,Weather_Temperature_Celsius,Weather_Relative_Humidity,Global_Horizontal_Radiation,Diffuse_Horizontal_Radiation,Wind_Direction,Weather_Daily_Rainfall,Radiation_Global_Tilted,Radiation_Diffuse_Tilted,Active_Power_Clean"
"2024-01-01 00:00:00,,,,,,,,,,,,,"
"2024-01-01 00:05:00,405221.0,0.30000004172325,0.010000000707805,,20.740343093872,18.584463119507,1.7161151170731,1.2969869375229,44.296714782715,0.0,,,0.010000000707805"


In [3]:
# Naive read -> everything ends up in ONE column (the problem)
print(pd.read_csv(RAW / "solar_dataset.csv", nrows=3).shape)

# Fixed loader (src/io_utils.py): strip the outer quotes, then parse
prim = load_primary_raw()
print(prim.shape)
prim.head()

(3, 1)


(223595, 13)


,Active_Energy_Delivered_Received,Current_Phase_Average,Active_Power,Wind_Speed,Weather_Temperature_Celsius,Weather_Relative_Humidity,Global_Horizontal_Radiation,Diffuse_Horizontal_Radiation,Wind_Direction,Weather_Daily_Rainfall,Radiation_Global_Tilted,Radiation_Diffuse_Tilted,Active_Power_Clean
timestamp,,,,,,,,,,,,,
2024-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-01-01 00:05:00,405221.0,0.3,0.0100,NaN,20.740343,18.584463,1.716115,1.296987,44.296715,0.0,NaN,NaN,0.010000
2024-01-01 00:10:00,NaN,NaN,NaN,NaN,20.562437,18.965721,1.737423,1.339449,45.068665,0.0,NaN,NaN,0.009167
2024-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.830827,0.929267,0.008333
2024-01-01 00:20:00,NaN,0.3,0.0075,NaN,20.331274,18.677074,1.791579,1.389650,44.926125,0.0,3.675133,0.812759,0.007500


In [4]:
print("Time span:", prim.index.min(), "->", prim.index.max())
print("Duplicate timestamps:", prim.index.duplicated().sum())
prim.info()

Time span: 2024-01-01 00:00:00 -> 2026-02-19 23:55:00
Duplicate timestamps: 0
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 223595 entries, 2024-01-01 00:00:00 to 2026-02-19 23:55:00
Data columns (total 13 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   Active_Energy_Delivered_Received  189698 non-null  float64
 1   Current_Phase_Average             189766 non-null  float64
 2   Active_Power                      189766 non-null  float64
 3   Wind_Speed                        0 non-null       float64
 4   Weather_Temperature_Celsius       186376 non-null  float64
 5   Weather_Relative_Humidity         186383 non-null  float64
 6   Global_Horizontal_Radiation       186409 non-null  float64
 7   Diffuse_Horizontal_Radiation      186408 non-null  float64
 8   Wind_Direction                    186408 non-null  float64
 9   Weather_Daily_Rainfall            186408 non-null  float64
 10  Radi

### 1.1 Sampling interval
Almost all steps are 5 minutes. The few 10 and 15 minute steps are missing rows.

In [5]:
prim.index.to_series().diff().value_counts().head()

timestamp
0 days 00:05:00    223413
0 days 00:10:00       157
0 days 00:15:00        11
0 days 00:20:00         4
2 days 06:25:00         1
Name: count, dtype: int64

### 1.2 Missing values and the longest gap per column

In [6]:
def longest_nan_run(s):
    """Length of the longest consecutive run of NaN values."""
    isna = s.isna()
    return int(isna.groupby((~isna).cumsum()).sum().max())

def missing_table(df, step_minutes):
    t = pd.DataFrame({
        "missing_%": df.isna().mean().mul(100).round(2),
        "longest_gap_rows": [longest_nan_run(df[c]) for c in df.columns],
    })
    t["longest_gap_hours"] = (t["longest_gap_rows"] * step_minutes / 60).round(1)
    return t

missing_table(prim, 5)

,missing_%,longest_gap_rows,longest_gap_hours
Active_Energy_Delivered_Received,15.16,862,71.8
Current_Phase_Average,15.13,862,71.8
Active_Power,15.13,862,71.8
Wind_Speed,100.00,223595,18632.9
Weather_Temperature_Celsius,16.65,6058,504.8
Weather_Relative_Humidity,16.64,6058,504.8
Global_Horizontal_Radiation,16.63,6058,504.8
Diffuse_Horizontal_Radiation,16.63,6058,504.8
Wind_Direction,16.63,6058,504.8
Weather_Daily_Rainfall,16.63,6058,504.8


**Observations**
- `Wind_Speed` is **100 % empty**, so it is dropped.
- About 15 % of power values are missing. The longest gap is about 72 h (power) and about 21 days (weather).

In [7]:
prim.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Active_Energy_Delivered_Received,189698.0,447389.26,23603.92,0.00,426396.25,448587.00,467804.75,487325.00
Current_Phase_Average,189766.0,6.44,8.39,0.00,0.30,0.31,13.08,33.71
Active_Power,189766.0,4.41,5.98,0.00,0.01,0.01,9.16,23.97
Wind_Speed,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Weather_Temperature_Celsius,186376.0,22.33,9.88,-3.35,15.29,22.84,29.84,44.28
Weather_Relative_Humidity,186383.0,39.70,25.03,0.00,19.74,32.02,57.12,102.91
Global_Horizontal_Radiation,186409.0,268.62,370.20,0.00,1.79,7.67,535.41,1503.90
Diffuse_Horizontal_Radiation,186408.0,58.11,96.39,0.00,1.29,7.07,77.62,757.74
Wind_Direction,186408.0,33.28,11.55,-12.24,29.34,33.52,37.12,181.02
Weather_Daily_Rainfall,186408.0,0.32,1.62,0.00,0.00,0.00,0.00,27.80


**Observations:** temperatures from −3 to 44 °C point to a desert climate. Humidity above 100 % and negative wind direction are sensor errors that will be fixed in cleaning. A GHI maximum of 1504 W/m² is above the clear-sky limit (cloud-enhancement events).

### 1.3 Is `Active_Power_Clean` safe to use as the target?

In [8]:
both = prim[["Active_Power", "Active_Power_Clean"]]
print("Rows where both exist and differ:", ((both.iloc[:, 0] - both.iloc[:, 1]).abs() > 1e-9).sum())
filled = prim["Active_Power"].isna() & prim["Active_Power_Clean"].notna()
print("Rows where raw power is missing but 'Clean' has a value:", filled.sum())

Rows where both exist and differ: 0
Rows where raw power is missing but 'Clean' has a value: 32901


`Active_Power_Clean` = raw power **plus 32,901 imputed values**. Training or evaluating on imputed values would reward a model for copying the imputation method. **Decision: use raw `Active_Power` as the target.**

### 1.4 Capacity and site location

In [9]:
p, g = prim["Active_Power"], prim["Global_Horizontal_Radiation"]
print(f"Estimated capacity (99.9th percentile of power): {p.quantile(0.999):.2f} kW (max {p.max():.2f})")
print(f"Correlation power vs GHI: {p.corr(g):.3f}")
print("Clock hour with highest mean GHI (solar noon):", g.groupby(prim.index.hour).mean().idxmax())
print("Month with highest mean GHI (summer):", g.groupby(prim.index.month).mean().idxmax())

daylight = (g > 5).groupby(prim.index.date).sum() * 5 / 60
daylight.index = pd.to_datetime(daylight.index)
daylight.groupby(daylight.index.month).median().round(1).rename("median daylight hours").to_frame().T

Estimated capacity (99.9th percentile of power): 19.13 kW (max 23.97)
Correlation power vs GHI: 0.995
Clock hour with highest mean GHI (solar noon): 12
Month with highest mean GHI (summer): 1


,1,2,3,4,5,6,7,8,9,10,11,12
median daylight hours,10.5,10.3,9.2,8.8,8.8,8.9,9.2,10.0,11.6,12.4,12.7,13.4


Summer falls in **January** and the longest days are in Dec–Jan, so the site is in the **southern hemisphere**. Solar noon is about 12:00 on the clock with no DST jump, so the timestamps are local standard time. The −3 to 44 °C range and the DKASC-style column names match **Alice Springs, Australia (−23.76°, 133.87°, UTC+9:30)**. *This is an assumption to confirm with the mentor.* The alignment with a solar model is verified in notebook 02.

## 2. Secondary dataset: `site_a_hourly_weather.csv`

In [10]:
site = load_site_a_raw()
print(site.shape, "|", site.index.min(), "->", site.index.max())
print("Duplicate timestamps:", site.index.duplicated().sum())
print(site.index.to_series().diff().value_counts().head(2))
site.head()

(13031, 21) | 2025-01-03 22:00:00+00:00 -> 2026-06-30 20:00:00+00:00
Duplicate timestamps: 0
datetime_utc
0 days 01:00:00    13030
Name: count, dtype: int64


,active_power_kw,dc_input_power_kw,daily_energy_kwh,cumulative_energy_kwh,inverter_temp_c,grid_freq_hz,power_factor,efficiency_pct,inverter_status,is_generating,temp_c,humidity_pct,cloud_cover_pct,wind_speed_kmh,ghi_wm2,dni_wm2,dhi_wm2,precip_mm,solar_elevation_deg,solar_azimuth_deg,capacity_factor
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2025-01-03 22:00:00+00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,359.0225,0.0
2025-01-03 23:00:00+00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.1,68.0,73.0,8.7,0.0,0.0,0.0,0.0,0.0,32.9826,0.0
2025-01-04 00:00:00+00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.6,63.0,46.0,9.1,0.0,0.0,0.0,0.0,0.0,57.1433,0.0
2025-01-04 01:00:00+00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-0.1,63.0,23.0,5.5,0.0,0.0,0.0,0.0,0.0,73.5177,0.0
2025-01-04 02:00:00+00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-0.8,67.0,53.0,5.2,0.0,0.0,0.0,0.0,0.0,85.8757,0.0


In [11]:
missing_table(site, 60)

,missing_%,longest_gap_rows,longest_gap_hours
active_power_kw,2.00,144,144.0
dc_input_power_kw,37.81,144,144.0
daily_energy_kwh,37.81,144,144.0
cumulative_energy_kwh,37.81,144,144.0
inverter_temp_c,37.81,144,144.0
grid_freq_hz,37.81,144,144.0
power_factor,37.81,144,144.0
efficiency_pct,37.81,144,144.0
inverter_status,37.81,144,144.0
is_generating,2.00,144,144.0


The inverter telemetry columns are **37.8 % empty**, so they are not used as model inputs. The weather and power columns are about 98 % complete.

In [12]:
site.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
active_power_kw,12771.0,9.30,14.74,0.00,0.00,0.29,13.21,55.00
dc_input_power_kw,8104.0,14.94,16.54,0.00,0.77,7.08,27.44,56.28
daily_energy_kwh,8104.0,119.26,124.03,0.00,6.33,71.44,217.54,421.85
cumulative_energy_kwh,8104.0,264358.89,34740.22,202230.87,233146.00,269707.33,288724.63,325989.39
inverter_temp_c,8104.0,32.08,15.02,-2.05,20.10,30.70,43.97,67.88
grid_freq_hz,8104.0,44.64,13.98,0.00,49.99,50.00,50.01,50.07
power_factor,8104.0,0.82,0.35,0.00,1.00,1.00,1.00,1.00
efficiency_pct,8104.0,78.89,34.69,0.00,85.37,97.86,98.22,98.58
is_generating,12771.0,0.56,0.50,0.00,0.00,1.00,1.00,1.00
temp_c,12769.0,11.24,8.99,-13.40,4.20,10.80,18.20,34.60


### 2.1 Location from the solar angles provided

In [13]:
el, az = site["solar_elevation_deg"], site["solar_azimuth_deg"]
print("Max solar elevation:", el.max(), "on", el.idxmax())
print("-> latitude approx", round(90 - el.max() + 23.44, 1), "deg N  (90 - elevation + 23.44 at the June solstice)")

near_noon = site[(az > 150) & (az < 210) & (el > 0)]
noon_utc = (near_noon.index.hour + (180 - near_noon["solar_azimuth_deg"]) / 15).groupby(near_noon.index.date).mean().mean()
print(f"Solar noon approx {noon_utc:.2f} h UTC -> longitude approx {(12 - noon_utc) * 15:.1f} deg E")

Max solar elevation: 68.4354 on 2025-06-21 10:00:00+00:00
-> latitude approx 45.0 deg N  (90 - elevation + 23.44 at the June solstice)
Solar noon approx 9.99 h UTC -> longitude approx 30.1 deg E


In [14]:
print(site["inverter_status"].value_counts(), "\n")
print("Hours at exactly 55 kW (inverter clipping):", (site["active_power_kw"] >= 55).sum())
print(f"Correlation power vs GHI: {site['active_power_kw'].corr(site['ghi_wm2']):.3f}")

inverter_status
Connected to grid                           6627
Standby: no sunlight                        1065
Standby: initializing                        204
Standby: detecting sunlight                  189
OFF: unexpected shutdown                      13
Standby: detecting insulation resistance       4
Standby: detecting grid power                  2
Name: count, dtype: int64 

Hours at exactly 55 kW (inverter clipping): 1
Correlation power vs GHI: 0.922


Site A is at about **45 °N, 30 °E** (Black Sea coast, Europe/Bucharest time with DST). It is a **55 kW** system whose inverter caps output at 55 kW. 13 hours are marked `OFF: unexpected shutdown`, which are faults and not weather.

## 3. Dataset decision

| | Primary (`solar_dataset.csv`) | Site A |
|---|---|---|
| Span | 26 months (2 full years) | 18 months |
| Resolution | 5 min → resample to hourly | hourly |
| Cloud cover | ✗ | ✓ |
| Target completeness | ~85 % | ~98 % |

- **Primary:** the main benchmark. Two full years mean the 20 % test set still has a full season, and yearly seasonality can be learnt.
- **Site A:** the second climate. It covers the cloud-cover research question and cross-site generalisation.